In [2]:
## Part B: local RAG song recommender
#- Dataset: `lewtun/music_genres_small` (1,000 songs from the Free Music Archive, 30 s each, 10 genres).
#- Retrieval: PANNs embeddings (same model as the demo) stored in a local Chroma database.
#- User preferences: songs the user likes (+ optional genres). Their taste = the average of those embeddings.
# Generation: gpt-4o-mini writes the recommendation using only the retrieved songs.

In [4]:
import os
import numpy as np
from dotenv import load_dotenv, find_dotenv
from IPython.display import Audio, display
from panns_inference import AudioTagging

_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
model = AudioTagging(checkpoint_path=None, device="cpu")

Checkpoint path: /Users/beholdingmars/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [2]:
from datasets import load_dataset, Audio

songs = load_dataset("lewtun/music_genres_small", split="train")
songs = songs.cast_column("audio", Audio(sampling_rate=32000))  # PANNs was trained on 32 kHz audio
songs

data/train-00000-of-00001-63d68663287b16(…): reconstructing file:   0%|          |  0.00B /  391MB            

data/train-00000-of-00001-63d68663287b16(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'song_id', 'genre_id', 'genre'],
    num_rows: 1000
})

In [5]:
from tqdm.auto import tqdm

SR = 32000
CLIP = 30 * SR   # every song cut or padded to exactly 30 s, so a batch is one rectangle

def to_fixed_length(wave, length=CLIP):
    wave = wave[:length]
    return np.pad(wave, (0, length - len(wave)))

embeddings = []
batch_size = 16
for start in tqdm(range(0, len(songs), batch_size)):
    batch = songs[start:start + batch_size]["audio"]
    waves = np.stack([to_fixed_length(a["array"]) for a in batch]).astype(np.float32)
    _, emb = model.inference(waves)
    embeddings.append(emb)
embeddings = np.vstack(embeddings)
embeddings.shape


  0%|          | 0/63 [00:00<?, ?it/s]

(1000, 2048)

In [7]:
!pip install -q chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2025.10.0 requires fsspec==2025.10.0, but you have fsspec 2025.3.0 which is incompatible.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.
streamlit 1.51.0 requires pyarrow<22,>=7.0, but you have pyarrow 26.0.0 which is incompatible.


In [8]:
import chromadb

client = chromadb.PersistentClient(path="./music_db")
collection = client.get_or_create_collection("songs", metadata={"hnsw:space": "cosine"})
collection.upsert(
    ids=[str(i) for i in range(len(songs))],
    embeddings=embeddings.tolist(),
    metadatas=[{"song_id": s, "genre": g} for s, g in zip(songs["song_id"], songs["genre"])],
)
collection.count()

1000

In [9]:
def recommend(liked, genres=None, k=5):
    """liked: row numbers of songs the user likes. genres: optional list of genres to stay inside."""
    taste = embeddings[liked].mean(axis=0)          # the user's "taste" = average of liked songs
    where = {"genre": {"$in": genres}} if genres else None
    res = collection.query(query_embeddings=[taste.tolist()], n_results=k + len(liked), where=where)
    picks = []
    for id_, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
        if int(id_) in liked:
            continue                                  # don't recommend what they already like
        picks.append({"row": int(id_), **meta, "similarity": round(1 - dist, 3)})
    return picks[:k]

liked = [i for i, g in enumerate(songs["genre"]) if g == "Punk"][:3]   # example user: likes 3 punk songs
recs = recommend(liked)
recs

[{'row': 617, 'genre': 'Punk', 'song_id': 16708, 'similarity': 0.951},
 {'row': 214, 'song_id': 10339, 'genre': 'Pop', 'similarity': 0.941},
 {'row': 662, 'genre': 'Punk', 'song_id': 18151, 'similarity': 0.938},
 {'row': 299, 'song_id': 9540, 'genre': 'Punk', 'similarity': 0.936},
 {'row': 811, 'song_id': 12340, 'genre': 'Rock', 'similarity': 0.934}]

In [10]:
from openai import OpenAI

llm = OpenAI(api_key=OPENAI_API_KEY)

def explain(liked, recs):
    liked_info = [{"row": i, "genre": songs["genre"][i]} for i in liked]
    prompt = (
        "You are a music recommender. Use ONLY the data below; do not invent titles or artists.\n"
        f"Songs the user likes: {liked_info}\n"
        f"Retrieved songs (similarity 0-1, by sound): {recs}\n"
        "Write a short recommendation: list each retrieved song by row and genre, and one line on why it fits."
    )
    r = llm.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print(explain(liked, recs))

Here are some song recommendations based on your preferences in punk music:

1. **Row 617 - Punk**  
   This song has a high similarity score of 0.951, making it a strong match for your taste in punk.

2. **Row 662 - Punk**  
   With a similarity score of 0.938, this track captures the raw energy and ethos of punk music that you enjoy.

3. **Row 299 - Punk**  
   This song, with a similarity score of 0.936, is sure to resonate with your appreciation for punk’s distinct sound.

4. **Row 214 - Pop**  
   Although it's a pop song, its similarity score of 0.941 suggests it has some elements that might appeal to you as a punk fan.

5. **Row 811 - Rock**  
   This rock song has a similarity score of 0.934, potentially offering a crossover vibe that punk enthusiasts can appreciate.

These selections should align well with your love of punk music!


In [11]:
def play(row):
    print(row, songs["genre"][row])
    display(Audio(songs[row]["audio"]["array"], rate=SR))

print("The user likes:")
for row in liked:
    play(row)
print("We recommend:")
for r in recs:
    play(r["row"])

The user likes:
7 Punk


14 Punk


17 Punk


We recommend:
617 Punk


214 Pop


662 Punk


299 Punk


811 Rock


In [ ]:
## Conclusions
#- What the recommender gets right (listen to B6): ...
#- Limitations: it compares sound, not lyrics or mood; the dataset has no titles or artists; the user is a fixed list of songs.
